# Week 2 - Logic in AI
## Lab Exercise - 1

In [371]:
import json
from kanren import run, facts, eq, Relation, var, conde

def parent(x,y):
    return conde ([father(x,y)], [mother(x,y)])
    
def grandparent(x,y):
    temp = var()
    return conde ((parent(x,temp), parent(temp,y)))
    
def sibling(x,y):
    temp = var()
    return conde ((parent(temp,x), parent(temp,y)))

# new spouse relationship
def spouse(x, y):
    return conde((father(x, c), mother(y, c)) | (father(y, c), mother(x, c)))

# new in_law relationships
def in_law(x, y):
    """
    Checks if x is an in-law of y (married to a sibling of y or vice versa).
    """
    temp = var()
    return conde((sibling(temp, y), spouse(x, temp)), (sibling(temp, x), spouse(y, temp)))

# def sibling_in_law(x, y):
#     """
#     Checks if x is a sibling-in-law of y (married to a sibling of y).
#     """
#     temp = var()
#     return conde((sibling(temp, y), spouse(x, temp)))

# def parent_in_law(x, y):
#     """
#     Checks if x is a parent-in-law of y (parent of y's spouse).
#     """
#     temp = var()
#     return conde((parent(x, temp), spouse(y, temp)), (parent(x, temp), spouse(temp, y)))
    
# new niece/nephew using sibling relationship - check if x is y's niece or nephew (child of a sibling)
def niece_nephew(x, y):
    temp1, temp2 = var(), var()
    return conde((sibling(temp1, y), child(temp1, temp2), eq(x, temp2)))

def uncle(x,y):
    temp = var()
    return conde ((father(temp,x), grandparent(temp,y)))

if __name__ == '__main__':
    father = Relation()
    mother = Relation()
    spouse = Relation()
    gender = Relation()
    
    with open('relationships.json') as f:
        d= json.loads(f.read())

    for item in d['father']:
        facts (father, (list(item.keys())[0],list(item.values())[0]))
        
    for item in d['mother']:
        facts (mother, (list(item.keys())[0],list(item.values())[0]))

    for item in d['spouse']:
        facts(spouse, (list(item.keys())[0], list(item.values())[0]))

    for item in d['gender']:
        facts(gender, (list(item.keys())[0], list(item.values())[0]))
        
    x= var()
    y = var()

In [306]:
#Finding father's children
name = 'John'
output = run(0,x,father(name,x))

print("Father " + name + "'s childern are : ")

for item in output: 
    print(item)

Father John's childern are : 
David
William
Adam


In [307]:
#Finding someone's parents
name = 'Adam'
output = run(0,x,parent(x,name))

print(name +"'s parents are :")

for item in output:
    print(item)

Adam's parents are :
John
Megan


In [308]:
#Finding grandparent's grandchildren 
name = 'Megan'
output = run(0,x,grandparent (name,x))
print(name +"'s grandchildren are:", output)

Megan's grandchildren are: ('Stephanie', 'Sophia', 'Julie', 'Chris', 'Neil', 'Tiffany', 'Peter', 'Wayne')


#### Question 1: write a script to find David's siblings

In [309]:
name = 'David'
siblings = set(run(0, x, sibling(x, name))) - {name}  # Convert to set and remove name
print(name + "'s siblings are:", siblings)

David's siblings are: {'William', 'Adam'}


#### Question 2: write a script to find Tiffany’s uncles

In [310]:
name = 'Tiffany' 
name_father = run(0, x, father(x, name))[0] 
output = run(0, x, uncle(x, name)) 
uncles = [x for x in output if x != name_father] 
print(name + "'s uncles are:", uncles) 

Tiffany's uncles are: ['Adam', 'William']


#### Question 3: write a script to find all spouses

In [311]:
seen_spouses = set()  # Create an empty set to store seen spouses

a, b, c = var(), var(), var()
output = run(0, (a, b), father(a, c), mother(b, c))

print("List of all spouses (Husband + Wife):")
for item in output:
  husband, wife = item[0], item[1]
  if husband not in seen_spouses and wife not in seen_spouses:
    seen_spouses.add(husband)
    seen_spouses.add(wife)
    print(husband,'+', wife)
print("\n")

# Define spouse as a Relation and Access the data directly from the new spouse dictionary loaded from the JSON file
for item in d['spouse']:
  person, spouse = list(item.keys())[0], list(item.values())[0]
  print(person, 'is married to', spouse)

List of all spouses (Husband + Wife):
William + Emma
Htut + Myat
John + Megan
David + Olivia
Adam + Lily


John is married to Megan
William is married to Emma
David is married to Olivia
Adam is married to Lily


Addtional Scripts

In [317]:
seen_spouses = set()

# Find all spouses
output = run(0, (a, b), spouse(a, b))
print("List of all spouses (Husband + Wife):")
for item in output:
  husband, wife = item[0], item[1]
  if husband not in seen_spouses and wife not in seen_spouses:
    seen_spouses.add(husband)
    seen_spouses.add(wife)
    print(husband, '+', wife)

List of all spouses (Husband + Wife):
David + Olivia
William + Emma
John + Megan
Adam + Lily


In [313]:
# David's niece/nephews 
name = "David"
seen = set()  # Create an empty set to store seen niece/nephews
output = run(0, x, niece_nephew(name, x))
print("List of " + name + "'s niece/nephews:")
for item in output:
  if item not in seen:
    seen.add(item)  # Add item to set if not seen before
    print(item)

List of David's niece/nephews:
Neil
Julie
Tiffany
Peter
Wayne


In [343]:
# Check if in-laws for yes condition 
name1 = "Steve"
name2 = "David"  # Assuming Bob is married to Alice's sibling

output = run(0, None, in_law(name1, name2))
if output:
  print(name1 + " and " + name2 + " are in-laws.")
else:
  print(name1 + " and " + name2 + " are not in-laws.")

Steve and David are in-laws.


In [333]:
# Check if in-laws for not condition 
name1 = "Sophia"
name2 = "David"  # Assuming Bob is married to Alice's sibling

output = run(0, None, in_law(name1, name2))
if output:
  print(name1 + " and " + name2 + " are in-laws.")
else:
  print(name1 + " and " + name2 + " are not in-laws.")

Sophia and David are not in-laws.


In [ ]:
"""
    name1 = "John"  # Assuming Michael is a parent
    name2 = "Olivia"  # Assuming Alice is married to Michael's child
    
    output = run(0, None, parent_in_law(name1, name2))
    if output:
      print(name1 + " is a parent-in-law of " + name2 + ".")
    else:
      print(name1 + " is not a parent-in-law of " + name2 + ".")
"""